<div style="font-size: 48px;">
v1_W4111-Fall-2025-002-HW4
</div>

# Introductions

This notebook is homework 4. It is a continuation of the previous notebook that tested your installation and ability to interact with MongoDB and Neo4j from a programming environment (Jupyter notebook). The homework is quite simple and is 3 relatively simple queries for MongoDB and 3 simple queries for Neo4j.

The homework is due on CourseWorks on 10-December at 11:59 PM.

The submission format is a PDF version of this notebook with the pages tagged/asigned to the questions.

# Setup

This section initializes and tests your environment. You do not have to reexecute these sections if you have completed and tested your environment and loaded the initial datasets. There is no harm running the cells a second time.

## General

In [7]:
import pandas
import json

## SQL

The homework is simpler than I had originally intended.

There is not reason to setup ipython-SQL, etc. So, I deleted those cells from a prior version of the notebook.

## MongoDB

In [1]:
# Since you have never used pymongo before, you will have to pip install it.
# You only need to do this the first time you execute the notebook.
# You can comment out the statement after the first time you run it.
# Since I have already installed pymongo, your output messages will be different from mine..
# As long as you do not have any errors, you are fine.
#
%pip install pymongo


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


[pymongo](https://pymongo.readthedocs.io/en/stable/) is the official connection library and software development kit (SDK) for interacting with MongoDB from Python environments.

In [2]:
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

# Initialize a connection to MongoDB through pymongo.
mongo_client = MongoClient()

try:
    # Run a simple command to verify connection.
    # As long as this command does not raise an exception, your installation is fine.
    result = mongo_client.admin.command("ping")
    print("Connected to MongoDB!")
except ConnectionFailure as e:
    print("Connection failed:", e)

Connected to MongoDB!


## Neo4j

In [3]:
# Since you have never used neo4j from Python before, you will have to pip install it.
# You only need to do this the first time you execute the notebook.
# You can comment out the statement after the first time you run it.
# Since I have already installed pymongo, your output messages will be different from mine..
# As long as you do not have any errors, you are fine.
#
%pip install neo4j


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


[neo4j](https://pypi.org/project/neo4j/) is the official Python driver and software development kit for Neo4j.

__Note__ You must change the password below to the value you selected when installing Neo4j and creating your first graph.

__Note__ This notebook assumes that you watched the recitation and followed the instructions to read/execute the Movie Graph/DB tutorial in Neo4j.

In [4]:
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable, AuthError

URI = "neo4j://localhost:7687"
USER = "neo4j"
PASSWORD = "dbuserdbuser"

def test_connection(uri, user, password):
    try:
        driver = GraphDatabase.driver(uri, auth=(user, password))

        # Try a simple query
        with driver.session() as session:
            result = session.run("RETURN 1 AS result")
            value = result.single()["result"]
            print("Connected! Test query returned:", value, ' which means success.')

        driver.close()

    except AuthError as e:
        print("Authentication failed:", e)
    except ServiceUnavailable as e:
        print("Cannot reach Neo4j:", e)
    except Exception as e:
        print("Unexpected error:", e)

if __name__ == "__main__":
    test_connection(URI, USER, PASSWORD)


Connected! Test query returned: 1  which means success.


# MongoDB Data Loading

Your MongoDB queries will use a subset [Game of Thrones data set.](https://github.com/jeffreylancaster/game-of-thrones) The necessary [JSON](https://www.json.org/json-en.html) input format files are in a subdirectory of the director containing this notebook.. JSON is a common data format for files containing nested, document like data.

In [5]:
# This function helps you with the data loading.
# It simpley open, reads and loads JSON data from a file.
#
def read_got_file(file_name, wrapper=None):
    #
    # Load JSON data from file at the file_name, which may include a relative path.
    # Some GoT files wrap the data in a top-level element.
    #
    result = None

    # Open the file for read.
    with open (file_name, "r") as in_file:

        # Parse and load the JSON data into Python data structures
        # using a standard library.
        result = json.load(in_file)

        # If there is a top-level element
        if wrapper:
            # Return the wrapped data.
            result = result[wrapper]
            
    return result
            

In [8]:
# Read the information about characters.
#
characters = read_got_file("data/GoT/characters.json", "characters")

In [9]:
characters[0:2]

[{'characterName': 'Addam Marbrand',
  'characterLink': '/character/ch0305333/',
  'actorName': 'B.J. Hogg',
  'actorLink': '/name/nm0389698/'},
 {'characterName': 'Aegon Targaryen',
  'houseName': 'Targaryen',
  'royal': True,
  'parents': ['Elia Martell', 'Rhaegar Targaryen'],
  'siblings': ['Rhaenys Targaryen', 'Jon Snow'],
  'killedBy': ['Gregor Clegane']}]

In [10]:
# Read and initialize the information about Game of Thrones episodes.
#
episodes = read_got_file("data/GoT/episodes.json", "episodes")

In [11]:
episodes[0:2]

[{'seasonNum': 1,
  'episodeNum': 1,
  'episodeTitle': 'Winter Is Coming',
  'episodeLink': '/title/tt1480055/',
  'episodeAirDate': '2011-04-17',
  'episodeDescription': "Jon Arryn, the Hand of the King, is dead. King Robert Baratheon plans to ask his oldest friend, Eddard Stark, to take Jon's place. Across the sea, Viserys Targaryen plans to wed his sister to a nomadic warlord in exchange for an army.",
  'openingSequenceLocations': ["King's Landing",
   'Winterfell',
   'The Wall',
   'Pentos'],
  'scenes': [{'sceneStart': '0:00:40',
    'sceneEnd': '0:01:45',
    'location': 'The Wall',
    'subLocation': 'Castle Black',
    'characters': [{'name': 'Gared'},
     {'name': 'Waymar Royce'},
     {'name': 'Will'}]},
   {'sceneStart': '0:01:45',
    'sceneEnd': '0:03:24',
    'location': 'North of the Wall',
    'subLocation': 'The Haunted Forest',
    'characters': [{'name': 'Gared'},
     {'name': 'Waymar Royce'},
     {'name': 'Will'}]},
   {'sceneStart': '0:03:24',
    'sceneEnd': 

In [12]:
# In case you run the notebook more than once, drop and reload the database you will use.
#
mongo_client.drop_database("F25_GoT")

In [13]:
# Create and load the collection characters in the database F25_GoT.
#
result = mongo_client["F25_GoT"]["characters"].insert_many(characters)

In [14]:
# The insert_many operation returns the MongoDB generated object IDs for each inserted document.
# Let's see how many we inserted.
len(result.inserted_ids)

389

In [15]:
# Create and load the collection characters in the database F25_GoT.
#
result = mongo_client["F25_GoT"]["episodes"].insert_many(episodes)

In [16]:
# The insert_many operation returns the MongoDB generated object IDs for each inserted document.
# Let's see how many we inserted.
len(result.inserted_ids)

73

# Neo4j Loading

Examining the information in ```characters``` indicates that there is embedded information about relationships between characters.

In [17]:
characters[0:3]

[{'characterName': 'Addam Marbrand',
  'characterLink': '/character/ch0305333/',
  'actorName': 'B.J. Hogg',
  'actorLink': '/name/nm0389698/',
  '_id': ObjectId('692b53cc9fd1297f98d55c83')},
 {'characterName': 'Aegon Targaryen',
  'houseName': 'Targaryen',
  'royal': True,
  'parents': ['Elia Martell', 'Rhaegar Targaryen'],
  'siblings': ['Rhaenys Targaryen', 'Jon Snow'],
  'killedBy': ['Gregor Clegane'],
  '_id': ObjectId('692b53cc9fd1297f98d55c84')},
 {'characterName': 'Aeron Greyjoy',
  'houseName': 'Greyjoy',
  'characterImageThumb': 'https://images-na.ssl-images-amazon.com/images/M/MV5BNzI5MDg0ZDAtN2Y2ZC00MzU1LTgyYjQtNTBjYjEzODczZDVhXkEyXkFqcGdeQXVyNTg0Nzg4NTE@._V1._SX100_SY140_.jpg',
  'characterImageFull': 'https://images-na.ssl-images-amazon.com/images/M/MV5BNzI5MDg0ZDAtN2Y2ZC00MzU1LTgyYjQtNTBjYjEzODczZDVhXkEyXkFqcGdeQXVyNTg0Nzg4NTE@._V1_.jpg',
  'characterLink': '/character/ch0540081/',
  'actorName': 'Michael Feast',
  'actorLink': '/name/nm0269923/',
  'siblings': ['Balon G

I examined the input data and determined that that relationship types are:

In [18]:
relationship_types = [
 'abducted',
 'abductedBy',
 'allies',
 'guardedBy',
 'guardianOf',
 'killed',
 'killedBy',
 'marriedEngaged',
 'parentOf',
 'parents',
 'servedBy',
 'serves',
 'sibling',
 'siblings'
]

I am going to process the ```characters``` information to extract relationships into a list of dictionaries of the form:
```
{
    "sourceCharacter": "sourceCharacterName", 
    "relationship": "relationshipLabel", 
    "targetCharacter": "targetCharacterName"
}
```

In [19]:
# The list of relationship dictionaries.
relationships = []

# Loop through the characters.
for c in characters:

    # For each type of relationship.
    for r in relationship_types:

        # Get the list of characters related by the relationship.
        rs = c.get(r, None)

        # Are there related characters?
        if rs:

            # For each character name related by the relationship.
            for rr in rs:

                # Make a new dictionary entry.
                new_r = {
                    "sourceCharacter": c["characterName"],
                    "relationship": r,
                    "targetCharacter": rr
                }

                # Add to the list of characters and relationships.
                relationships.append(new_r)
                        

In [20]:
relationships[0:5]

[{'sourceCharacter': 'Aegon Targaryen',
  'relationship': 'killedBy',
  'targetCharacter': 'Gregor Clegane'},
 {'sourceCharacter': 'Aegon Targaryen',
  'relationship': 'parents',
  'targetCharacter': 'Elia Martell'},
 {'sourceCharacter': 'Aegon Targaryen',
  'relationship': 'parents',
  'targetCharacter': 'Rhaegar Targaryen'},
 {'sourceCharacter': 'Aegon Targaryen',
  'relationship': 'siblings',
  'targetCharacter': 'Rhaenys Targaryen'},
 {'sourceCharacter': 'Aegon Targaryen',
  'relationship': 'siblings',
  'targetCharacter': 'Jon Snow'}]

In [21]:
len(relationships)

847

To enable you to run the notebook multiple times, we will delete any previously loaded data.

In [22]:
# You must set the password to the password you used when installing Neo4j Desktop.
#
from neo4j import GraphDatabase

URI = "neo4j://localhost:7687"
USER = "neo4j"
PASSWORD = "dbuserdbuser"



In [23]:
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

def delete_relationship(tx):
    # relationship type must be injected into query string, not as a parameter
    query = f"""
    MATCH (c:Character)-[r]-(c2:Character) DELETE r
    """
    tx.run(query)

def delete_character(tx):
    # relationship type must be injected into query string, not as a parameter
    query = f"""
    MATCH (c:Character) DELETE c
    """
    tx.run(query)

with driver.session() as session:
    session.execute_write(delete_relationship)
    session.execute_write(delete_character)

driver.close()

It is now time to load the characters and their relationships.

In [24]:
from neo4j import GraphDatabase

URI = "neo4j://localhost:7687"
USER = "neo4j"
PASSWORD = "dbuserdbuser"

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))


def add_relationship(tx, source, rel, target):
    # relationship type must be injected into query string, not as a parameter
    query = f"""
    MERGE (src:Character {{name: $source}})
    MERGE (tgt:Character {{name: $target}})
    MERGE (src)-[:{rel.upper()}]->(tgt)
    """
    tx.run(query, source=source, target=target)

for data in relationships:
    with driver.session() as session:
        session.execute_write(
            add_relationship,
            data["sourceCharacter"],
            data["relationship"],
            data["targetCharacter"],
        )
        print("Added", data)

driver.close()


Added {'sourceCharacter': 'Aegon Targaryen', 'relationship': 'killedBy', 'targetCharacter': 'Gregor Clegane'}
Added {'sourceCharacter': 'Aegon Targaryen', 'relationship': 'parents', 'targetCharacter': 'Elia Martell'}
Added {'sourceCharacter': 'Aegon Targaryen', 'relationship': 'parents', 'targetCharacter': 'Rhaegar Targaryen'}
Added {'sourceCharacter': 'Aegon Targaryen', 'relationship': 'siblings', 'targetCharacter': 'Rhaenys Targaryen'}
Added {'sourceCharacter': 'Aegon Targaryen', 'relationship': 'siblings', 'targetCharacter': 'Jon Snow'}
Added {'sourceCharacter': 'Aeron Greyjoy', 'relationship': 'siblings', 'targetCharacter': 'Balon Greyjoy'}
Added {'sourceCharacter': 'Aeron Greyjoy', 'relationship': 'siblings', 'targetCharacter': 'Euron Greyjoy'}
Added {'sourceCharacter': 'Aerys II Targaryen', 'relationship': 'killed', 'targetCharacter': 'Brandon Stark'}
Added {'sourceCharacter': 'Aerys II Targaryen', 'relationship': 'killed', 'targetCharacter': 'Rickard Stark'}
Added {'sourceCharac

# Some Queries

## MongoDB

### Characters and Scenes

This query (a MongoDB [aggregation](https://www.mongodb.com/docs/manual/aggregation/)):
1. Unwinds episodes $\rightarrow$ scenes $\rightarrow$ characters into a flattened data structure.
2. Assign a number to each scene starting at 1 within an episode.
3. Converts the text representation of sceneStart and scenedEnd into seconds as integer.
4. "Joins" with characters to get the name of the actor, if it is in the dataset.

In [25]:
# Requires the PyMongo package.
# https://api.mongodb.com/python/current

client = MongoClient('mongodb://localhost:27017/')
result = client['F25_GoT']['episodes'].aggregate([
    {
        '$unwind': {
            'path': '$scenes', 
            'includeArrayIndex': 'sceneNo'
        }
    }, {
        '$project': {
            'seasonNum': 1, 
            'episodeNum': 1, 
            'sceneNo': {
                '$add': [
                    '$sceneNo', 1
                ]
            }, 
            'sceneStart': '$scenes.sceneStart', 
            'sceneEnd': '$scenes.sceneEnd', 
            'location': '$scenes.location', 
            'subLocation': '$scenes.subLocation', 
            'characters': '$scenes.characters'
        }
    }, {
        '$unwind': {
            'path': '$characters'
        }
    }, {
        '$project': {
            'seasonNum': 1, 
            'episodeNum': 1, 
            'sceneNo': 1, 
            'sceneStart': 1, 
            'sceneEnd': 1, 
            'location': 1, 
            'subLocation': 1, 
            'characterName': '$characters.name'
        }
    }, {
        '$addFields': {
            'sceneStart': {
                '$let': {
                    'vars': {
                        'parts': {
                            '$split': [
                                '$sceneStart', ':'
                            ]
                        }
                    }, 
                    'in': {
                        '$add': [
                            {
                                '$multiply': [
                                    {
                                        '$toInt': {
                                            '$arrayElemAt': [
                                                '$$parts', 0
                                            ]
                                        }
                                    }, 3600
                                ]
                            }, {
                                '$multiply': [
                                    {
                                        '$toInt': {
                                            '$arrayElemAt': [
                                                '$$parts', 1
                                            ]
                                        }
                                    }, 60
                                ]
                            }, {
                                '$toInt': {
                                    '$arrayElemAt': [
                                        '$$parts', 2
                                    ]
                                }
                            }
                        ]
                    }
                }
            }, 
            'sceneEnd': {
                '$let': {
                    'vars': {
                        'parts': {
                            '$split': [
                                '$sceneEnd', ':'
                            ]
                        }
                    }, 
                    'in': {
                        '$add': [
                            {
                                '$multiply': [
                                    {
                                        '$toInt': {
                                            '$arrayElemAt': [
                                                '$$parts', 0
                                            ]
                                        }
                                    }, 3600
                                ]
                            }, {
                                '$multiply': [
                                    {
                                        '$toInt': {
                                            '$arrayElemAt': [
                                                '$$parts', 1
                                            ]
                                        }
                                    }, 60
                                ]
                            }, {
                                '$toInt': {
                                    '$arrayElemAt': [
                                        '$$parts', 2
                                    ]
                                }
                            }
                        ]
                    }
                }
            }
        }
    }, {
        '$lookup': {
            'from': 'characters', 
            'localField': 'characterName', 
            'foreignField': 'characterName', 
            'as': 'characterInfo'
        }
    }, {
        '$project': {
            'seasonNum': 1, 
            'episodeNum': 1, 
            'sceneNo': 1, 
            'sceneStart': 1, 
            'sceneEnd': 1, 
            'location': 1, 
            'subLocation': 1, 
            'characterName': 1, 
            'actorName': {
                '$arrayElemAt': [
                    '$characterInfo.actorName', 0
                ]
            }, 
            'characterInfo': 1
        }
    }, {
        '$project': {
            'seasonNum': 1, 
            'episodeNum': 1, 
            'sceneNo': 1, 
            'sceneStart': 1, 
            'sceneEnd': 1, 
            'location': 1, 
            'subLocation': 1, 
            'characterName': 1, 
            'actorName': 1, 
            'characterInfo': 1
        }
    }
])

In [26]:
result = list(result)

In [27]:
result_df = pandas.DataFrame(result)

In [29]:
result_df

,_id,seasonNum,episodeNum,sceneNo,sceneStart,sceneEnd,location,subLocation,characterName,characterInfo,actorName
0,692b53d29fd1297f98d55e08,1,1,1,40,105,The Wall,Castle Black,Gared,"[{'_id': 692b53cc9fd1297f98d55cc7, 'characterN...",Dermot Keaney
1,692b53d29fd1297f98d55e08,1,1,1,40,105,The Wall,Castle Black,Waymar Royce,"[{'_id': 692b53cc9fd1297f98d55deb, 'characterN...",Rob Ostlere
2,692b53d29fd1297f98d55e08,1,1,1,40,105,The Wall,Castle Black,Will,"[{'_id': 692b53cc9fd1297f98d55df3, 'characterN...",Bronson Webb
3,692b53d29fd1297f98d55e08,1,1,2,105,204,North of the Wall,The Haunted Forest,Gared,"[{'_id': 692b53cc9fd1297f98d55cc7, 'characterN...",Dermot Keaney
4,692b53d29fd1297f98d55e08,1,1,2,105,204,North of the Wall,The Haunted Forest,Waymar Royce,"[{'_id': 692b53cc9fd1297f98d55deb, 'characterN...",Rob Ostlere
...,...,...,...,...,...,...,...,...,...,...,...
12109,692b53d29fd1297f98d55e50,8,6,90,4700,4780,North of the Wall,The Wall,Tormund Giantsbane,"[{'_id': 692b53cc9fd1297f98d55ddc, 'characterN...",Kristofer Hivju
12110,692b53d29fd1297f98d55e50,8,6,90,4700,4780,North of the Wall,The Wall,Ghost,"[{'_id': 692b53cc9fd1297f98d55ccb, 'characterN...",NaN
12111,692b53d29fd1297f98d55e50,8,6,91,4780,4820,North of the Wall,The Haunted Forest,Jon Snow,"[{'_id': 692b53cc9fd1297f98d55cfe, 'characterN...",Kit Harington
12112,692b53d29fd1297f98d55e50,8,6,91,4780,4820,North of the Wall,The Haunted Forest,Tormund Giantsbane,"[{'_id': 692b53cc9fd1297f98d55ddc, 'characterN...",Kristofer Hivju


### Create a View

Let's assume we plan to reuse the result of the query many times. We can create a view using the aggregation pipeline.

In [30]:
pipeline = [
    {
        '$unwind': {
            'path': '$scenes', 
            'includeArrayIndex': 'sceneNo'
        }
    }, {
        '$project': {
            'seasonNum': 1, 
            'episodeNum': 1, 
            'sceneNo': {
                '$add': [
                    '$sceneNo', 1
                ]
            }, 
            'sceneStart': '$scenes.sceneStart', 
            'sceneEnd': '$scenes.sceneEnd', 
            'location': '$scenes.location', 
            'subLocation': '$scenes.subLocation', 
            'characters': '$scenes.characters'
        }
    }, {
        '$unwind': {
            'path': '$characters'
        }
    }, {
        '$project': {
            'seasonNum': 1, 
            'episodeNum': 1, 
            'sceneNo': 1, 
            'sceneStart': 1, 
            'sceneEnd': 1, 
            'location': 1, 
            'subLocation': 1, 
            'characterName': '$characters.name'
        }
    }, {
        '$addFields': {
            'sceneStart': {
                '$let': {
                    'vars': {
                        'parts': {
                            '$split': [
                                '$sceneStart', ':'
                            ]
                        }
                    }, 
                    'in': {
                        '$add': [
                            {
                                '$multiply': [
                                    {
                                        '$toInt': {
                                            '$arrayElemAt': [
                                                '$$parts', 0
                                            ]
                                        }
                                    }, 3600
                                ]
                            }, {
                                '$multiply': [
                                    {
                                        '$toInt': {
                                            '$arrayElemAt': [
                                                '$$parts', 1
                                            ]
                                        }
                                    }, 60
                                ]
                            }, {
                                '$toInt': {
                                    '$arrayElemAt': [
                                        '$$parts', 2
                                    ]
                                }
                            }
                        ]
                    }
                }
            }, 
            'sceneEnd': {
                '$let': {
                    'vars': {
                        'parts': {
                            '$split': [
                                '$sceneEnd', ':'
                            ]
                        }
                    }, 
                    'in': {
                        '$add': [
                            {
                                '$multiply': [
                                    {
                                        '$toInt': {
                                            '$arrayElemAt': [
                                                '$$parts', 0
                                            ]
                                        }
                                    }, 3600
                                ]
                            }, {
                                '$multiply': [
                                    {
                                        '$toInt': {
                                            '$arrayElemAt': [
                                                '$$parts', 1
                                            ]
                                        }
                                    }, 60
                                ]
                            }, {
                                '$toInt': {
                                    '$arrayElemAt': [
                                        '$$parts', 2
                                    ]
                                }
                            }
                        ]
                    }
                }
            }
        }
    }, {
        '$lookup': {
            'from': 'characters', 
            'localField': 'characterName', 
            'foreignField': 'characterName', 
            'as': 'characterInfo'
        }
    }, {
        '$project': {
            'seasonNum': 1, 
            'episodeNum': 1, 
            'sceneNo': 1, 
            'sceneStart': 1, 
            'sceneEnd': 1, 
            'location': 1, 
            'subLocation': 1, 
            'characterName': 1, 
            'actorName': {
                '$arrayElemAt': [
                    '$characterInfo.actorName', 0
                ]
            }, 
            'characterInfo': 1
        }
    }, {
        '$project': {
            'seasonNum': 1, 
            'episodeNum': 1, 
            'sceneNo': 1, 
            'sceneStart': 1, 
            'sceneEnd': 1, 
            'location': 1, 
            'subLocation': 1, 
            'characterName': 1, 
            'actorName': 1, 
            'characterInfo': 1
        }
    }
]

In [31]:
mongo_client["F25_GoT"].create_collection(
    "characters_scenes",
    viewOn="episodes",
    pipeline=pipeline
)

print("View created.")

View created.


Test the view by finding Robb Stark.

In [32]:
result = mongo_client["F25_GoT"]["characters_scenes"].find(
    {'characterName': 'Robb Stark'},
    {'seasonNum': 1, 'episodeNum': 1, 'sceneNo': 1, 'sceneStart': 1, 'sceneEnd': 1, 'characterName': 1, 'actorName': 1}
)
    

In [33]:
result_df = pandas.DataFrame(list(result))
result_df

,_id,seasonNum,episodeNum,sceneNo,sceneStart,sceneEnd,characterName,actorName
0,692b53d29fd1297f98d55e08,1,1,14,567,758,Robb Stark,Richard Madden
1,692b53d29fd1297f98d55e08,1,1,15,758,941,Robb Stark,Richard Madden
2,692b53d29fd1297f98d55e08,1,1,16,941,1124,Robb Stark,Richard Madden
3,692b53d29fd1297f98d55e08,1,1,20,1389,1419,Robb Stark,Richard Madden
4,692b53d29fd1297f98d55e08,1,1,31,2654,2848,Robb Stark,Richard Madden
...,...,...,...,...,...,...,...,...
83,692b53d29fd1297f98d55e24,3,9,68,2804,2832,Robb Stark,Richard Madden
84,692b53d29fd1297f98d55e24,3,9,69,2832,2918,Robb Stark,Richard Madden
85,692b53d29fd1297f98d55e24,3,9,70,2918,2936,Robb Stark,Richard Madden
86,692b53d29fd1297f98d55e25,3,10,4,168,189,Robb Stark,Richard Madden


### Characters Time on Screen

Use the view to compute the total time a character was on screen for the entire series.

In [56]:
pipeline = [
    {
        '$project': {
            'characterName': 1, 
            'sceneLength': {
                '$subtract': [
                    '$sceneEnd', '$sceneStart'
                ]
            }
        }
    }, {
        '$group': {
            '_id': '$characterName',
            'screenTime': {
                '$sum': '$sceneLength'
            }
        }
    }, {
        '$sort': {
            'screenTime': -1
        }
    },
    {
        '$project': {
            '_id': 0,
            'characterName': '$_id', 
            'screenTime': 1
        }
    }
]

In [57]:
result = client['F25_GoT']['characters_scenes'].aggregate(pipeline)
result_df = pandas.DataFrame(list(result))

In [58]:
result_df

,screenTime,characterName
0,41104,Tyrion Lannister
1,40365,Jon Snow
2,31694,Daenerys Targaryen
3,25705,Sansa Stark
4,25522,Cersei Lannister
...,...,...
572,15,Vayon Poole
573,15,Rickard Stark
574,14,Tyrell Guard
575,14,Olly's Mother


## Neo4j Queries

### Find Tom Hanks

__Note__ This test assumes that you watched the recitation and ran the Movie Graph/DB tutorial.

In [59]:
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

def node_to_dict(node):
    return {
        "element_id": node.element_id,
        "labels": list(node.labels),
        **dict(node.items())
    }

def get_person_dict(name):
    query = """
    MATCH (p:Person {name: $name})
    RETURN p
    """
    with driver.session() as session:
        record = session.run(query, name=name).single()
        if record:
            return node_to_dict(record["p"])
        return None


person = get_person_dict("Tom Hanks")
print(person)

driver.close()



{'element_id': '4:28ff268f-c812-4554-960c-edec73840f4a:71', 'labels': ['Person'], 'born': 1956, 'name': 'Tom Hanks'}


### Movies Associated with Tom Hanks

In [60]:
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

query = """
MATCH (p:Person {name: $name})-[r]->(m:Movie)
RETURN p.name AS name, type(r) AS relType, m.title AS title
"""

with driver.session() as session:
    result = session.run(query, name="Tom Hanks")
    rows = [record.data() for record in result]

for r in rows:
    n = r["name"]
    rel = r["relType"]
    t = r["title"]
    f = f'{n} {rel} "{t}"'
    print(f)


Tom Hanks ACTED_IN "You've Got Mail"
Tom Hanks ACTED_IN "Sleepless in Seattle"
Tom Hanks ACTED_IN "Joe Versus the Volcano"
Tom Hanks ACTED_IN "That Thing You Do"
Tom Hanks ACTED_IN "Cloud Atlas"
Tom Hanks ACTED_IN "The Da Vinci Code"
Tom Hanks ACTED_IN "The Green Mile"
Tom Hanks ACTED_IN "Apollo 13"
Tom Hanks ACTED_IN "Cast Away"
Tom Hanks ACTED_IN "Charlie Wilson's War"
Tom Hanks ACTED_IN "The Polar Express"
Tom Hanks ACTED_IN "A League of Their Own"
Tom Hanks DIRECTED "That Thing You Do"


### Also Starred

In [61]:
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

query = """
MATCH (p:Person {name: $name})-[r]->(m:Movie)<-[:ACTED_IN]-(p2:Person)
RETURN p.name AS name, type(r) AS relType, m.title AS title, p2.name as costar
"""

with driver.session() as session:
    result = session.run(query, name="Tom Hanks")
    rows = [record.data() for record in result]

for r in rows:
    n = r["name"]
    rel = r["relType"]
    t = r["title"]
    c = r['costar']
    f = f'{n} {rel} "{t}" costarring {c}'
    print(f)

Tom Hanks ACTED_IN "You've Got Mail" costarring Meg Ryan
Tom Hanks ACTED_IN "You've Got Mail" costarring Greg Kinnear
Tom Hanks ACTED_IN "You've Got Mail" costarring Parker Posey
Tom Hanks ACTED_IN "You've Got Mail" costarring Dave Chappelle
Tom Hanks ACTED_IN "You've Got Mail" costarring Steve Zahn
Tom Hanks ACTED_IN "Sleepless in Seattle" costarring Meg Ryan
Tom Hanks ACTED_IN "Sleepless in Seattle" costarring Rita Wilson
Tom Hanks ACTED_IN "Sleepless in Seattle" costarring Bill Pullman
Tom Hanks ACTED_IN "Sleepless in Seattle" costarring Victor Garber
Tom Hanks ACTED_IN "Sleepless in Seattle" costarring Rosie O'Donnell
Tom Hanks ACTED_IN "Joe Versus the Volcano" costarring Meg Ryan
Tom Hanks ACTED_IN "Joe Versus the Volcano" costarring Nathan Lane
Tom Hanks ACTED_IN "That Thing You Do" costarring Charlize Theron
Tom Hanks ACTED_IN "That Thing You Do" costarring Liv Tyler
Tom Hanks ACTED_IN "Cloud Atlas" costarring Hugo Weaving
Tom Hanks ACTED_IN "Cloud Atlas" costarring Halle Berry


### And Directed By

In [62]:
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

query = """
MATCH (p:Person {name: $name})-[r]->(m:Movie)<-[:ACTED_IN]-(p2:Person), (m:Movie)<-[:DIRECTED]-(d:Person)
RETURN p.name AS name, type(r) AS relType, m.title AS title, p2.name as costar, d.name as director
"""

with driver.session() as session:
    result = session.run(query, name="Tom Hanks")
    rows = [record.data() for record in result]

for r in rows:
    n = r["name"]
    rel = r["relType"]
    t = r["title"]
    c = r['costar']
    d = r['director']
    f = f'{n} {rel} "{t}" costarring {c} directed by {d}'
    print(f)

Tom Hanks ACTED_IN "You've Got Mail" costarring Meg Ryan directed by Nora Ephron
Tom Hanks ACTED_IN "You've Got Mail" costarring Greg Kinnear directed by Nora Ephron
Tom Hanks ACTED_IN "You've Got Mail" costarring Parker Posey directed by Nora Ephron
Tom Hanks ACTED_IN "You've Got Mail" costarring Dave Chappelle directed by Nora Ephron
Tom Hanks ACTED_IN "You've Got Mail" costarring Steve Zahn directed by Nora Ephron
Tom Hanks ACTED_IN "Sleepless in Seattle" costarring Meg Ryan directed by Nora Ephron
Tom Hanks ACTED_IN "Sleepless in Seattle" costarring Rita Wilson directed by Nora Ephron
Tom Hanks ACTED_IN "Sleepless in Seattle" costarring Bill Pullman directed by Nora Ephron
Tom Hanks ACTED_IN "Sleepless in Seattle" costarring Victor Garber directed by Nora Ephron
Tom Hanks ACTED_IN "Sleepless in Seattle" costarring Rosie O'Donnell directed by Nora Ephron
Tom Hanks ACTED_IN "Joe Versus the Volcano" costarring Meg Ryan directed by John Patrick Stanley
Tom Hanks ACTED_IN "Joe Versus th

# Student Homework Questions

## Introduction

## MongoDB

### M1

Set the ```filter``` and ```projection``` to find the character "Arys Stark" and return only the ```characterName``` and the list of people she ```killed.```

See my answer below before running your code. Your answer should match mine.

In [63]:
filter = {}
projection = {
}

In [64]:
result = mongo_client["F25_GoT"]["characters"].find(
    filter,
    projection
)
result = list(result)
result
    

[{'_id': ObjectId('692b53cc9fd1297f98d55c92'),
  'characterName': 'Arya Stark',
  'killed': ['Red Keep Stableboy',
   'Frey Soldier #1',
   'Polliver',
   'Rorge',
   'Ghita',
   'Meryn Trant',
   'The Waif',
   'Black Walder Rivers',
   'Lothar Frey',
   'Walder Frey',
   'Petyr Baelish',
   'The Night King',
   'White Walker',
   'Viserion']}]

### M2

Set the ```filter``` and ```projection``` to return the ```characterName``` and list of characters ```killed``` for people who have ```killed  "White Walker."

See my answer below before running your code. Your answer should match mine.

In [65]:
filter = {}
projection = {
}

In [66]:
result = mongo_client["F25_GoT"]["characters"].find(
    filter,
    projection
)
result = list(result)
result

[{'_id': ObjectId('692b53cc9fd1297f98d55c92'),
  'characterName': 'Arya Stark',
  'killed': ['Red Keep Stableboy',
   'Frey Soldier #1',
   'Polliver',
   'Rorge',
   'Ghita',
   'Meryn Trant',
   'The Waif',
   'Black Walder Rivers',
   'Lothar Frey',
   'Walder Frey',
   'Petyr Baelish',
   'The Night King',
   'White Walker',
   'Viserion']},
 {'_id': ObjectId('692b53cc9fd1297f98d55cfe'),
  'characterName': 'Jon Snow',
  'killed': ['Othor',
   'Qhorin Halfhand',
   'Orell',
   'Karl Tanner',
   'Styr',
   'Mance Rayder',
   'Janos Slynt',
   'White Walker',
   'Alliser Thorne',
   'Othell Yarwyck',
   'Bowen Marsh',
   'Olly',
   'Lyanna Stark',
   'Daenerys Targaryen']},
 {'_id': ObjectId('692b53cc9fd1297f98d55d50'),
  'characterName': 'Meera Reed',
  'killed': ['Jojen Reed', 'White Walker']},
 {'_id': ObjectId('692b53cc9fd1297f98d55dad'),
  'characterName': 'Samwell Tarly',
  'killed': ['White Walker', 'Thenn Warg']}]

### M3

This question is a little tricky and requires writing an [aggregation pipeline.](https://www.mongodb.com/docs/manual/core/aggregation-pipeline/)  The easiest way to write the pipeline is:
- Use the aggregation creation tool in Compass.
- Copy the pipeline over to the cell below.

You will have to look up some of the pipeline operators and do a little investigation of your own. You can also use ChatGPT, Claude Code, etc. to help you if you want.

The pipeline:
- "Selects" all characters with ```characterName``` ending in " Stark."
- Gets the ```characterName``` and list of characters the person killed.
- "Flattens" or "Unwinds" the result to have one entry of the form ```(characterName, killedCharacterName)``` for each character killed.

See my answer below before running your code. Your answer should match mine.

In [69]:
pipeline = [
]

In [70]:
result = mongo_client['F25_GoT']['characters'].aggregate(
    pipeline
)
result_df = pandas.DataFrame(list(result))
result_df


     

,characterName,killed
0,Arya Stark,Red Keep Stableboy
1,Arya Stark,Frey Soldier #1
2,Arya Stark,Polliver
3,Arya Stark,Rorge
4,Arya Stark,Ghita
5,Arya Stark,Meryn Trant
6,Arya Stark,The Waif
7,Arya Stark,Black Walder Rivers
8,Arya Stark,Lothar Frey
9,Arya Stark,Walder Frey


## Neo4j

### N1

In the 

```
query = """
... ...
"""
``` 
below, enter a Cypher query that finds the ```title``` of every ```Movie``` that "Tom Hanks" OR "Tom Cruise" ```ACTED_IN```. 

See my answer below before running your code. Your answer should match mine.

In [72]:
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

query = """
... ...
"""

with driver.session() as session:
    result = session.run(query, name="Tom Hanks")
    rows = [record.data() for record in result]

for r in rows:
    n = r["name"]
    t = r["title"]
    f = f'{n} acted in "{t}"'
    print(f)

Tom Cruise acted in "A Few Good Men"
Tom Cruise acted in "Top Gun"
Tom Cruise acted in "Jerry Maguire"
Tom Hanks acted in "You've Got Mail"
Tom Hanks acted in "Sleepless in Seattle"
Tom Hanks acted in "Joe Versus the Volcano"
Tom Hanks acted in "That Thing You Do"
Tom Hanks acted in "Cloud Atlas"
Tom Hanks acted in "The Da Vinci Code"
Tom Hanks acted in "The Green Mile"
Tom Hanks acted in "Apollo 13"
Tom Hanks acted in "Cast Away"
Tom Hanks acted in "Charlie Wilson's War"
Tom Hanks acted in "The Polar Express"
Tom Hanks acted in "A League of Their Own"


### N2

In the 

```
query = """
... ...
"""
``` 
below, enter a Cypher query that finds the ```title``` of every ```Movie``` that "Tom Hanks" ```DIRECTED``` and the ```name``` of very ```Person``` that ```ACTED_IN``` one of the movies. 

See my answer below before running your code. Your answer should match mine.

In [74]:
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

query = """
... ...
"""

with driver.session() as session:
    result = session.run(query, name="Tom Hanks")
    rows = [record.data() for record in result]

for r in rows:
    n = r["name"]
    t = r["title"]
    c = r['actor']
    f = f'{n} directed "{t}" starring {c}'
    print(f)

Tom Hanks directed "That Thing You Do" starring Charlize Theron
Tom Hanks directed "That Thing You Do" starring Tom Hanks
Tom Hanks directed "That Thing You Do" starring Liv Tyler


### N3

In the 

```
query = """
... ...
"""
``` 
below, enter a Cypher query that finds the ```name``` of every ```Character``` with name ending in " Stark" who ```KILLED``` another ```Character``` and also finds the ```Characters``` that ```Character``` ```KILLED``` returns the ```name```.

See my answer below before running your code. Your answer should match mine.

In [76]:
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

query = """
... ...
"""

with driver.session() as session:
    result = session.run(query)
    rows = [record.data() for record in result]

for r in rows:
    n = r["name"]
    n1 = r["n1"]
    n2 = r['n2']
    f = f'{n} killed {n1} who in turn killed {n2}'
    print(f)

Arya Stark killed Polliver who in turn killed Lommy Greenhands
Arya Stark killed Meryn Trant who in turn killed Syrio Forel
Arya Stark killed The Waif who in turn killed Lady Crane
Arya Stark killed Black Walder Rivers who in turn killed Catelyn Stark
Arya Stark killed Lothar Frey who in turn killed Talisa Maegyr
Arya Stark killed Petyr Baelish who in turn killed Lysa Arryn
Arya Stark killed Petyr Baelish who in turn killed Joffrey Baratheon
Arya Stark killed Petyr Baelish who in turn killed Dontos Hollard
Arya Stark killed The Night King who in turn killed Viserion
Arya Stark killed The Night King who in turn killed Theon Greyjoy
Arya Stark killed The Night King who in turn killed Ned Umber
Arya Stark killed The Night King who in turn killed Three-Eyed Raven
Arya Stark killed White Walker who in turn killed Loboda
Arya Stark killed Viserion who in turn killed Pyat Pree
Arya Stark killed Viserion who in turn killed Great Master #1
Robb Stark killed Rickard Karstark who in turn killed M